# Exploring Time Series Datasets

## Learning Objectives

By completing this notebook you will:
1. Load and inspect three real-world time series datasets
2. Apply a consistent EDA workflow: shape, dtypes, time range, missing values, seasonality
3. Recognize the nixtla `(unique_id, ds, y)` format in diverse dataset types
4. Identify key properties that affect forecasting model choice

**Estimated time:** 13 minutes  
**Datasets:** French Bakery (retail), ETTm1 (energy), Blog Traffic (web analytics)

---

In [ ]:
learning_objectives([
    "13 minutes",
    "French Bakery (retail), ETTm1 (energy), Blog Traffic (web analytics)"
])

## Setup

Import all libraries in one cell. `statsmodels` provides the seasonal decomposition function we use for seasonality analysis. Everything else is standard.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose

plt.style.use('seaborn-v0_8-whitegrid')

print("Setup complete.")

## Dataset 1: French Bakery Daily Sales

**What it is:** Daily unit sales of 8 bakery items (baguette, croissant, pain au chocolat, etc.) from a French bakery chain.

**Why it matters for forecasting:**
- Strong weekly seasonality (weekends sell more)
- No missing dates
- Multiple correlated series — cross-learning helps
- Short enough to train quickly on a laptop

In [ ]:
# Load French Bakery daily sales
bakery_url = (
    "https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series/"
    "main/datasets/french_bakery_daily.csv"
)
bakery = pd.read_csv(bakery_url, parse_dates=['ds'])

print("=== French Bakery ===")
print(f"Shape       : {bakery.shape}")
print(f"Columns     : {bakery.columns.tolist()}")
print(f"Dtypes      : {dict(bakery.dtypes)}")
print(f"Series      : {sorted(bakery['unique_id'].unique())}")
print(f"Date range  : {bakery['ds'].min().date()} to {bakery['ds'].max().date()}")
print(f"Missing y   : {bakery['y'].isna().sum()}")
print()
print("Sample rows:")
print(bakery.head(3).to_string(index=False))

In [ ]:
# Summary statistics per series
bakery_stats = (
    bakery.groupby('unique_id')['y']
    .agg(['count', 'mean', 'std', 'min', 'max'])
    .rename(columns={'count': 'n_obs', 'mean': 'avg_sales', 'std': 'std_sales'})
    .round(1)
)
print("Per-series statistics:")
print(bakery_stats.to_string())

In [ ]:
# Seasonal decomposition for baguette — reveals weekly pattern
baguette = bakery[bakery['unique_id'] == 'baguette'].sort_values('ds').set_index('ds')

# Additive decomposition: y = trend + seasonal + residual
# period=7 for weekly seasonality in daily data
decomp = seasonal_decompose(baguette['y'], model='additive', period=7, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
decomp.observed.plot(ax=axes[0], color='black', linewidth=0.8)
axes[0].set_ylabel('Observed')

decomp.trend.plot(ax=axes[1], color='steelblue', linewidth=1.2)
axes[1].set_ylabel('Trend')

decomp.seasonal.iloc[:14].plot(ax=axes[2], color='darkorange', linewidth=1.2)
axes[2].set_ylabel('Seasonal (weekly)')

decomp.resid.plot(ax=axes[3], color='grey', linewidth=0.6)
axes[3].set_ylabel('Residual')

plt.suptitle('Baguette Daily Sales — Additive Seasonal Decomposition', y=1.01)
plt.tight_layout()
plt.show()

# What fraction of variance does the weekly component explain?
seasonal_var = decomp.seasonal.var()
total_var = baguette['y'].var()
print(f"Weekly seasonality explains {100 * seasonal_var / total_var:.1f}% of total variance")

**Observation:** The weekly seasonal component accounts for a substantial fraction of total variance. A model that does not learn this pattern (e.g., a model with `input_size < 7`) will produce systematically biased forecasts.

---

## Dataset 2: ETTm1 (Electricity Transformer Temperature — 15-minute)

**What it is:** Oil temperature and load measurements from electricity transformers in China, recorded every 15 minutes over 2 years.

**Why it matters for forecasting:**
- High frequency (15-minute intervals)
- Multiple seasonal periods: daily (96 steps), weekly (672 steps)
- Standard benchmark for long-horizon forecasting research
- Demonstrates how nixtla format works for non-retail data

In [ ]:
# ETTm1 is available directly from the original repository
ettm1_url = (
    "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/"
    "main/ETT-small/ETTm1.csv"
)
ettm1_raw = pd.read_csv(ettm1_url, parse_dates=['date'])

print("=== ETTm1 (raw) ===")
print(f"Shape   : {ettm1_raw.shape}")
print(f"Columns : {ettm1_raw.columns.tolist()}")
print(f"Date range: {ettm1_raw['date'].min()} to {ettm1_raw['date'].max()}")
print()
print(ettm1_raw.head(3).to_string(index=False))

In [ ]:
# Convert wide format → nixtla long format (unique_id, ds, y)
# Each column (HUFL, HULL, MUFL, ..., OT) becomes a separate series
value_cols = [c for c in ettm1_raw.columns if c != 'date']

ettm1 = ettm1_raw.melt(
    id_vars='date',
    value_vars=value_cols,
    var_name='unique_id',
    value_name='y',
).rename(columns={'date': 'ds'})

print("=== ETTm1 (nixtla format) ===")
print(f"Shape      : {ettm1.shape}")
print(f"Series     : {sorted(ettm1['unique_id'].unique())}")
print(f"Date range : {ettm1['ds'].min()} to {ettm1['ds'].max()}")
print(f"Missing y  : {ettm1['y'].isna().sum()}")
print()
print(ettm1.head(3).to_string(index=False))

In [ ]:
# The target variable is OT (Oil Temperature)
# Plot 2 weeks of OT data to see the daily pattern
ot = ettm1[ettm1['unique_id'] == 'OT'].sort_values('ds')

# Subset: 2 weeks = 14 days * 96 steps/day = 1344 rows
two_weeks = ot.head(1344)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(two_weeks['ds'], two_weeks['y'], linewidth=0.8, color='darkorange')
ax.set_title('ETTm1: Oil Temperature — First 2 Weeks (15-minute intervals)')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(f"Steps per day : 96  (24 hours × 4 per hour)")
print(f"Steps per week: {96 * 7}")
print(f"Total steps   : {len(ot):,}")
print(f"Total duration: {(ot['ds'].max() - ot['ds'].min()).days} days")

In [ ]:
# EDA: compare all ETTm1 series
ettm1_stats = (
    ettm1.groupby('unique_id')['y']
    .agg(['count', 'mean', 'std', 'min', 'max'])
    .rename(columns={'count': 'n_obs', 'mean': 'avg', 'std': 'std_dev'})
    .round(2)
)
print("ETTm1 series statistics:")
print(ettm1_stats.to_string())

print()
print("Forecasting implications:")
print("  - High-frequency data: input_size needs to cover at least 1 full day (96 steps)")
print("  - OT is the standard target variable in ETT benchmark papers")
print("  - HUFL/HULL/MUFL/MULL are load features — can be used as covariates")

---

## Dataset 3: Blog Traffic

**What it is:** Daily page views for 145,063 blog posts collected from a blogging platform (derived from the Kaggle Web Traffic Time Series Forecasting competition).

**Why it matters for forecasting:**
- Intermittent demand: many days with zero views
- High skewness: most posts have low views; viral posts have millions
- Large scale: demonstrates neuralforecast's cross-learning across thousands of series
- Irregular start dates: series have different lengths

In [ ]:
# Blog traffic subset — M4 weekly competition data is a good proxy
# We use a subset available from nixtla's datasets repo
blog_url = (
    "https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series/"
    "main/datasets/blog_data.csv"
)

try:
    blog = pd.read_csv(blog_url, parse_dates=['ds'])
    print("Loaded from nixtla transfer-learning repository.")
except Exception:
    # Fallback: use M4 weekly data from datasetsforecast
    from datasetsforecast.m4 import M4
    blog, _, _ = M4.load(directory='/tmp/m4', group='Weekly')
    print("Loaded M4 Weekly as blog traffic proxy.")

print(f"\nShape      : {blog.shape}")
print(f"Columns    : {blog.columns.tolist()}")
print(f"Series     : {blog['unique_id'].nunique():,}")
print(f"Date range : {blog['ds'].min().date()} to {blog['ds'].max().date()}")
print(f"Missing y  : {blog['y'].isna().sum():,}")

In [ ]:
# Distribution of series lengths — are they all the same?
series_lengths = blog.groupby('unique_id').size()

print("Series length distribution:")
print(series_lengths.describe().round(0))

fig, ax = plt.subplots(figsize=(10, 4))
series_lengths.hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of observations per series')
ax.set_ylabel('Count of series')
ax.set_title('Distribution of Series Lengths')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of y values — shows skewness typical of web traffic
y_nonzero = blog[blog['y'] > 0]['y']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw scale — dominated by large outliers
axes[0].hist(y_nonzero.clip(upper=y_nonzero.quantile(0.99)),
             bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('y distribution (clipped at 99th pct)')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

# Log scale — more informative for skewed data
axes[1].hist(np.log1p(y_nonzero), bins=50, color='darkorange', edgecolor='white')
axes[1].set_title('log(1 + y) distribution')
axes[1].set_xlabel('log(1 + value)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Traffic Data: Raw vs Log-Transformed Distribution', y=1.01)
plt.tight_layout()
plt.show()

print(f"Skewness of raw y     : {y_nonzero.skew():.2f}")
print(f"Skewness of log(1+y)  : {np.log1p(y_nonzero).skew():.2f}")
print()
print("Forecasting implication: use scaler_type='robust' or log-transform y")
print("before fitting to prevent a few viral posts from dominating gradients.")

---

## Comparing the Three Datasets

Each dataset illustrates a different forecasting challenge. The table below summarizes the key properties that determine which models and configurations work best.

In [ ]:
# Build a comparison summary
comparison = pd.DataFrame([
    {
        'Dataset': 'French Bakery',
        'Frequency': 'Daily',
        'N series': bakery['unique_id'].nunique(),
        'Rows/series': bakery.groupby('unique_id').size().median(),
        'Seasonality': 'Weekly (period=7)',
        'Missing values': bakery['y'].isna().sum(),
        'Skewness': round(bakery['y'].skew(), 2),
        'Best for': 'Learning the API',
    },
    {
        'Dataset': 'ETTm1',
        'Frequency': '15-minute',
        'N series': ettm1['unique_id'].nunique(),
        'Rows/series': ettm1.groupby('unique_id').size().median(),
        'Seasonality': 'Daily (period=96), weekly (period=672)',
        'Missing values': ettm1['y'].isna().sum(),
        'Skewness': round(ettm1['y'].skew(), 2),
        'Best for': 'Long-horizon benchmarks',
    },
    {
        'Dataset': 'Blog / M4 Weekly',
        'Frequency': 'Weekly',
        'N series': blog['unique_id'].nunique(),
        'Rows/series': blog.groupby('unique_id').size().median(),
        'Seasonality': 'Annual (period=52)',
        'Missing values': blog['y'].isna().sum(),
        'Skewness': round(blog['y'].skew(), 2),
        'Best for': 'Cross-learning at scale',
    },
])

print(comparison.to_string(index=False))

## The Nixtla Format: A Checklist

Before passing any DataFrame to `NeuralForecast`, verify all five conditions:

In [ ]:
def validate_nixtla_format(df: pd.DataFrame, name: str = 'dataset') -> bool:
    """Check that a DataFrame meets the nixtla (unique_id, ds, y) contract."""
    checks = []

    # 1. Required columns present
    required = {'unique_id', 'ds', 'y'}
    has_cols = required.issubset(df.columns)
    checks.append(("Required columns present", has_cols,
                   f"Missing: {required - set(df.columns)}" if not has_cols else ""))

    # 2. ds is datetime
    is_datetime = pd.api.types.is_datetime64_any_dtype(df['ds'])
    checks.append(("ds is datetime type", is_datetime,
                   "Convert with pd.to_datetime(df['ds'])" if not is_datetime else ""))

    # 3. y is numeric
    is_numeric = pd.api.types.is_numeric_dtype(df['y'])
    checks.append(("y is numeric type", is_numeric,
                   "Convert y to float" if not is_numeric else ""))

    # 4. No duplicate (unique_id, ds) pairs
    n_dupes = df.duplicated(subset=['unique_id', 'ds']).sum()
    checks.append(("No duplicate (unique_id, ds) pairs", n_dupes == 0,
                   f"{n_dupes} duplicates found" if n_dupes > 0 else ""))

    # 5. No missing y values
    n_missing = df['y'].isna().sum()
    checks.append(("No missing y values", n_missing == 0,
                   f"{n_missing} NaN values — forward-fill or drop" if n_missing > 0 else ""))

    print(f"\n=== Validating: {name} ===")
    all_passed = True
    for label, passed, hint in checks:
        status = "PASS" if passed else "FAIL"
        line = f"  [{status}] {label}"
        if hint:
            line += f"  ->  {hint}"
        print(line)
        if not passed:
            all_passed = False

    print(f"  Result: {'All checks passed' if all_passed else 'Fix the issues above'}")
    return all_passed


validate_nixtla_format(bakery, name='French Bakery')
validate_nixtla_format(ettm1,  name='ETTm1')
validate_nixtla_format(blog,   name='Blog / M4 Weekly')

In [ ]:
section_divider("Summary")

## Summary

### What you explored

| Dataset | Key property | Forecasting challenge |
|---|---|---|
| French Bakery | Weekly seasonality, 8 series | Seasonal pattern learning |
| ETTm1 | 15-min frequency, multi-seasonal | Long horizon, high frequency |
| Blog traffic | Skewed, high series count | Scale, cross-learning |

### EDA checklist for any new time series dataset
1. Check shape, dtypes, missing values
2. Identify the time range and frequency
3. Count unique series and their lengths
4. Examine distribution of `y` (raw and log-scaled)
5. Run seasonal decomposition to identify dominant period
6. Validate the nixtla format before fitting any model

### What's next
- **Module 01:** Point forecasting — train NHITS and NBEATS on ETTm1, compare against baselines
- **Module 02:** Probabilistic forecasting — add `MQLoss` and evaluate calibration on bakery data
- **Guide 02:** NeuralForecast ecosystem — `.cross_validation()`, `.simulate()`, `.explain()`

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])